In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from transformers import CLIPProcessor, CLIPModel
from peft import PeftModel
from PIL import Image
import pandas as pd
import os
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
import gc


class Config:
    teacher_base_model = "openai/clip-vit-base-patch32"
    teacher_adapter_path = "clip_v1_sports_finished" 
    
    student_model_name = "mobilenet_v3_large"
    
    data_path = "final_dataset_v3_ready.csv"
    img_dir = "data/data/train"
    output_dir = "mobilenet_v3_distilled_student"
    
    batch_size = 32     
    epochs = 10
    learning_rate = 1e-4 

    temperature = 4.0    
    alpha = 0.5         
    device = "cuda" if torch.cuda.is_available() else "cpu"

print(f" Ready for Knowledge Distillation on {Config.device}")


class DistillationDataset(Dataset):
    def __init__(self, df, img_dir, clip_processor, mobile_transform):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.clip_processor = clip_processor
        self.mobile_transform = mobile_transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['filename'])
        
        label = 0 if row['final_label'] in ['Safe_General', 'Safe_Contextual_Body'] else 1

        try:
            image = Image.open(img_path).convert("RGB")
            

            teacher_input = self.clip_processor(images=image, return_tensors="pt")['pixel_values'].squeeze(0)
            
            student_input = self.mobile_transform(image)
            
            return teacher_input, student_input, torch.tensor(label, dtype=torch.long)
        except:
            return None, None, None

def collate_fn(batch):
    batch = [b for b in batch if b[0] is not None]
    if len(batch) == 0: return None
    t_imgs = torch.stack([b[0] for b in batch])
    s_imgs = torch.stack([b[1] for b in batch])
    labels = torch.stack([b[2] for b in batch])
    return t_imgs, s_imgs, labels


def distillation_loss(student_logits, teacher_logits, labels, T, alpha):
    hard_loss = F.cross_entropy(student_logits, labels)
    

    distillation_loss = nn.KLDivLoss(reduction="batchmean")(
        F.log_softmax(student_logits / T, dim=1),
        F.softmax(teacher_logits / T, dim=1)
    ) * (T * T) 
    total_loss = alpha * distillation_loss + (1.0 - alpha) * hard_loss
    return total_loss

def train_distillation():
    gc.collect()
    torch.cuda.empty_cache()
    
    base_model = CLIPModel.from_pretrained(Config.teacher_base_model)
    teacher_model = PeftModel.from_pretrained(base_model, Config.teacher_adapter_path)
    teacher_model = teacher_model.merge_and_unload()
    
    teacher_model.to(Config.device)
    teacher_model.eval() 
    processor = CLIPProcessor.from_pretrained(Config.teacher_base_model)
    
    print("Pre-computing Teacher's Text Embeddings...")
    text_prompts = ["Safe content", "Unsafe content"]
    text_inputs = processor(text=text_prompts, return_tensors="pt", padding=True).to(Config.device)
    with torch.no_grad():
        teacher_text_features = teacher_model.get_text_features(**text_inputs)
        teacher_text_features = teacher_text_features / teacher_text_features.norm(dim=1, keepdim=True)

    print("Initializing Student (MobileNetV3)...")
    student_model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V1)
    student_model.classifier[3] = nn.Linear(student_model.classifier[3].in_features, 2)
    student_model.to(Config.device)
    

    student_transforms = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(20),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    df = pd.read_csv(Config.data_path)
    train_df, val_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df['final_label']) 
    
    train_loader = DataLoader(
        DistillationDataset(train_df, Config.img_dir, processor, student_transforms),
        batch_size=Config.batch_size, shuffle=True, collate_fn=collate_fn
    )
    
    val_transforms = transforms.Compose([
        transforms.Resize((224, 224)), transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    val_loader = DataLoader(
        DistillationDataset(val_df, Config.img_dir, processor, val_transforms),
        batch_size=Config.batch_size, shuffle=False, collate_fn=collate_fn
    )

    optimizer = optim.AdamW(student_model.parameters(), lr=Config.learning_rate)
    
    print("Class is in session! (Distillation Started)")
    best_acc = 0.0
    
    for epoch in range(Config.epochs):
        student_model.train()
        total_loss = 0
        
        pbar = tqdm(train_loader, desc=f"Ep {epoch+1}/{Config.epochs}")
        for teacher_imgs, student_imgs, labels in pbar:
            teacher_imgs = teacher_imgs.to(Config.device)
            student_imgs = student_imgs.to(Config.device)
            labels = labels.to(Config.device)
            
            with torch.no_grad():
                img_feats = teacher_model.get_image_features(teacher_imgs)
                img_feats = img_feats / img_feats.norm(dim=1, keepdim=True)
                teacher_logits = 100.0 * img_feats @ teacher_text_features.T
            
            student_logits = student_model(student_imgs)
            
            loss = distillation_loss(
                student_logits, teacher_logits, labels, 
                Config.temperature, Config.alpha
            )
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            pbar.set_postfix({"Distill Loss": f"{loss.item():.4f}"})
            
        student_model.eval()
        correct = 0; total = 0
        with torch.no_grad():
            for _, s_imgs, lbls in val_loader: 
                s_imgs, lbls = s_imgs.to(Config.device), lbls.to(Config.device)
                outputs = student_model(s_imgs)
                preds = outputs.argmax(dim=1)
                correct += (preds == lbls).sum().item()
                total += lbls.size(0)
        
        acc = 100 * correct / total
        print(f" Ep {epoch+1} Avg Loss: {total_loss/len(train_loader):.4f} |  Student Acc: {acc:.2f}%")
        
        if acc > best_acc:
            best_acc = acc
            torch.save(student_model.state_dict(), f"{Config.output_dir}_best.pth")
            print(" Smart Student Model Saved!")
    print(f" Final Distilled Accuracy: {best_acc:.2f}%")
    return student_model

model = train_distillation()

In [ ]:
import os
import torch
import torch.nn as nn
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from torchvision import models, transforms

MODEL_PATH = "mobilenet_v3_distilled_student_best.pth" 
TARGET_FOLDER = "test/unsafe/" 
device = "cuda" if torch.cuda.is_available() else "cpu"

def load_mobilenet_model():
    print(f"Loading MobileNetV3 from {MODEL_PATH}...")
    try:
        model = models.mobilenet_v3_large(weights=None)

        in_features = model.classifier[3].in_features
        model.classifier[3] = nn.Linear(in_features, 2)
        
        model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
        model.to(device)
        model.eval() 

        print("Model loaded successfully.")
        return model
    except Exception as e:
        print(f"Error loading model: {e}")
        return None

inference_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

CLASSES = ["Safe", "Unsafe"]

def scan_folder(folder_path, model):
    valid_extensions = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')
    files = [f for f in os.listdir(folder_path) if f.lower().endswith(valid_extensions)]
    
    if len(files) == 0:
        print(f"No valid images found in {folder_path}.")
        return

    print(f"Starting scan of {len(files)} images...")
    results = []
    
    for filename in tqdm(files, desc="Scanning"):
        img_path = os.path.join(folder_path, filename)
        
        try:
            image = Image.open(img_path).convert("RGB")
            # Add batch dimension
            input_tensor = inference_transforms(image).unsqueeze(0).to(device) 
            
            with torch.no_grad():
                outputs = model(input_tensor)
                probs = torch.softmax(outputs, dim=1).squeeze()
                
            top_prob, top_idx = probs.topk(1)
            confidence = top_prob.item() * 100
            verdict = CLASSES[top_idx.item()]
            
            results.append({
                "Filename": filename,
                "Verdict": verdict,
                "Confidence": f"{confidence:.2f}%",
                "Full Path": img_path
            })
            
            # Log unsafe predictions for monitoring
            if verdict == "Unsafe":
                tqdm.write(f"Alert - {filename}: {verdict} ({confidence:.1f}%)")
                
        except Exception as e:
            print(f"Error processing {filename}: {e}")

    # Save report
    df = pd.DataFrame(results)
    df.to_csv("mobilenet_scan_report.csv", index=False)
    
    print("\n" + "="*40)
    print("Scan Summary:")
    print(df['Verdict'].value_counts().to_string())
    print("="*40)

if __name__ == "__main__":
    model = load_mobilenet_model()
    if model:
        scan_folder(TARGET_FOLDER, model)